# Week 6: Experiments and Statistics

In this practical you will conduct experiments to tune the parameters of an EA, learn how to present the results and conduct some statistical testing to determine whether an operator has a signficant effect on the results.

In this tutorial you will learn:

1. How to design and conduct experiments to collect data to investigate the influence of parameters
2. How to calculate statistics from the data
3. How to plot the data using boxplots
4. How to conduct statistical testing using an appropriate test

It is also important that you are able to intepret the boxplots and statistical tests, and explain the observations  using your knowledge of EAs

We will use the EASimple to solve the One-Max problem as a test bed (as it is quick to run) to do multiple experiments and gather some statistics that will help us understand the influence of the various operators and tuning parameters

The code for plotting and calculating statistics can easily be re-used with any EA (e.g. can be useful for your coursework!)

## Libraries for plotting and calculating statistics

Before you start, you will need to make sure you have the  **pandas** library and the **scipy** installed in your system (if you are using Google colabs then this should be there already

In [ ]:
# Run this only  if you need to install pandas/scipy
!pip install pandas
!pip install scipy

## Set up

Let's first define the EA - this code should all look familiar by now

In [ ]:
#  Full code listing for EASimple to solve a OneMax problem

import random


from deap import base
from deap import creator
from deap import tools
from deap import algorithms

import pandas as pd
import scipy
import numpy as np
import matplotlib.pyplot as plt

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)


toolbox = base.Toolbox()

# random boolean values
toolbox.register("attr_bool", random.randint, 0, 1)

# one max - chromosomes length 100
toolbox.register("individual", tools.initRepeat, creator.Individual, 
    toolbox.attr_bool, 100)

# define the population to be a list of individuals
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


# the goal ('fitness') function to be maximized
def evalOneMax(individual):
    return sum(individual),

# initialse some default start parameters
MUTPB = 0.1
CXPB = 0.6
POPSIZE = 50
NGEN=200
TNSIZE=2


# register the goal / fitness function
toolbox.register("evaluate", evalOneMax)

# register the crossover operator
toolbox.register("mate", tools.cxOnePoint)

# register a mutation operator 
toolbox.register("mutate", tools.mutFlipBit, indpb=MUTPB)

# register crossove
toolbox.register("select", tools.selTournament, tournsize=TNSIZE)




In [ ]:
# define a main as usual
def runEA():
    
    # create a new population
    pop = toolbox.population(n=POPSIZE)
    
    # keep track of the single best solution found
    hof = tools.HallOfFame(1)
 
    # create a statistics object: we can log what ever statistics we want using this. We use the numpy Python library
    # to calculate the stats and label them with convenient labels
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # run the algorithm: 
    # cxpb = crossover probability; mutpb = mutation probability; ngen = number of iterations
    pop, log = algorithms.eaSimple(pop, toolbox, cxpb=CXPB, mutpb=MUTPB, ngen=NGEN, 
                                   stats=stats, halloffame=hof, verbose=False)
    
    return pop, log, hof



## How to use this code to collect statistics

Now we have written the EA, we can write some code to call it multiple times. In this example experiment, we create an experiment to investigate the influence of **population size**.  The EA is run for the POPSIZE parameter in a range from 50 to 151 in steps of 50 (i.e. tests 3 values 50,100,150).  For each value of POPSIZE, the EA is run 10 times so we can collate statistics.  

After each run of the algorithm, we can extract the statistics of interest from the logbook and then write them to a dataframe that is going to keep a record of the information of interest from each logbook.  Dataframes are convenient to work with in Python, particularly for collating statistics as we can group the data in any way that's convenient.

**In the first experiment we will investigate whether changing population size has an influence on performance**.

We will run the algorithm with POPSIZE set to values of 50-150 in steps of 50 (it's more interesting to try a wider range of values but takes longer to run of course, so for the purpose of illustration we will just look at a small range of values).  For each value of POPSIZE, we will run the algorithm 10 times.

- After each experiment (i.e. a single run of the EA for a given POPSIZE), we will extract the best fitness found and store in a dataframe. This is a convenient way to manage the data to extract statistics later
- we will also save the log file produced from each experiment as we can use this information to do some plotting later

Run the following - note it still may take some time to run on your machine

In [ ]:
# Experiment:  test POPSIZE from 50-150 in steps of 50

all_logs=[]     # make an empty list to store the log files prodiced

# create an dataframe that has 3 columns to record the important data from each run
df_Bests = pd.DataFrame({'popsize': pd.Series(dtype='int'),
                   'fitness': pd.Series(dtype='float'),
                   'genMaxFound': pd.Series(dtype='int')})

for  POPSIZE in range(50, 151, 50):  # from 50 to 150 in steps of 50
        # repeat EA 10x for each parameter
        for reps in range(10):
            pop,log,hof = runEA()
            # extract the best fitness
            best = hof[0].fitness.values[0]
            # save the generation this fitness was first found
            max = log.select('max')
            for gen in range(NGEN):  
                if max[gen] == best:
                    break   
            
            # update the df with the new data
            df_Bests = df_Bests.append({'popsize': POPSIZE , 'fitness': best, 'genMaxFound':gen}, ignore_index=True)
            
            # add the log which has the information about each generation  from each experiment to a list
            all_logs.append(log)

# Data
Have a look at the dataframe we have just created - you should see 3 columns showing the best fitness and the generation at which it was found for each run of the algorithm


You can find information about using *pandas* dataframes for plotting and calculating statistics on many data science sites/StackOverflow/Google.... feel free to explore... some useful things are explained in the rest of the notebook

In [ ]:
df_Bests

# Plotting aggregated data to show the evolutionary trajectory

As in the experiments we did in previous weeks, the logbooks contain the minimum, average and maximum values of fitness function at each generation of the algorithm.

We can aggregate this data over the 10 runs for a particular parameter.  The list *all_logs* contains 30 logs: 10 logs for each of the experiments associated with population size (50, 100, 150):

For example
- all_logs[0] is the log for the 1st run of popsize 50
- all_logs[1] is the log for the 1st run of popsize 50
- ....
- all_logs[10]  is the log for the 1st run of popsize 100
- all_logs[20]  is the log for the 1st run of popsize 100

etc.

For each value of POPSIZE, we can aggregate the log data over the 10 runs: e.g at generation 1 in each run, we can look up the **best fitness value (maximum)**.  This gives us 10 values from each of 10 runs at each generation.  We can calculate the average, minimum and maximum of these 10 values at each generation and plot this data


The code below explains how to extract the aggregated data. We can then plot the convergence curve that shows:

- the mean of the 10 values representing "max" fitness at each generation
- the "maximum" of the 10 values representing "max" fitness at each generation
- the "minumum" of the 10 values representing "max" fitness at each generation

This is a useful graph as it tells us something about how much the different runs vary: if the shaded band is very wide, it means the runs are very variable

In [ ]:


all_logs_df = []


# this loops over logs 0-9, i.e. the 10 logs associate with POPSIZE=50. You can change it for other POPSIZE values
for j in  range(0,10):
    logbook = all_logs[j]  # extract the relacant logbook 
    data = [[i for i in item.values()] for item in logbook]  # this creates a list [0,1,2...n] where n is the length of the log book - it is need for creating the x-axis when plotting
    header = list(logbook.header)
    df = pd.DataFrame(data, columns=header)  # turn the logbook into a dataframe as its easier to work with
    all_logs_df.append(df) # add the log a list of dataframes
       
       
df_result = pd.concat(all_logs_df)   # this makes one dataframe from the 10 separate ones we created above

#calculate the mean value of the 10 best results ("max") found at every generation in every run [i,e from 10 values]
#repeat to calculate the "minimum" value of the 10 best results at each generation
#repeat to calculate the "maximum" value of the 10 best results at each generation

max_mean=df_result['max'].groupby(df_result['gen']).apply(np.mean) # look at the max column, group results by "gen", and apply the relevant statistic
max_max=df_result['max'].groupby(df_result['gen']).apply(np.max)
max_min=df_result['max'].groupby(df_result['gen']).apply(np.min)


#plot convergence curves for data of choice

x = [i for i in range(0,201)]  # make sure this is set to NGEN+1 ... it creates a list [0,1,2,....NGEN] used to define the x-axis for plotting
fig, ax = plt.subplots()
# plot the mean value of the best result per generation
ax.plot(max_mean, color='red')
# not plot the max and min value of the best result per generation and shade to show the variance per run
ax.fill_between(x,max_min, max_max, alpha=0.2)





If you want to view the curves for POPSIZE=100 or POPSIZE=150 then you need to alter the indices in the loop at line 8 in the cell above (e.g. for 10,20 for POPSIZE 100 and 20,30 for POPSIZE 200)

Plot these curves too

## plotting std and mean

Instead of plotting the mean and using shading to show the max/min variation in values, we can plot the standard deviation at each point


In [ ]:

# show std instead of min/max variation
max_std=df_result['max'].groupby(df_result['gen']).apply(np.std)    # calculate standard dev of the values from each of the runs at each generation of the runs

max_max = max_mean+max_std
max_min = max_mean-max_std

x = [i for i in range(0,201)]  # make sure this is set to NGEN+1 ... it creates a list [0,1,2,....NGEN] used to define the x-axis for plotting
fig, ax = plt.subplots()
# plot the mean value of the best result per generation
ax.plot(max_mean, color='red')
# not plot the max and min value of the best result per generation and shade to show the variance per run
ax.fill_between(x,max_min, max_max, alpha=0.2)



# Plotting boxplots

Refer to the lecture for an explanation of a boxplot

Here we aggregate the data by the population size variable and plot the boxplot for each of the 3 values of popsize

Observe whether there is much overlap between the boxes and whether they suggest that one of the values for popsize might produce significantly better results than another 

Look at the second plot which shows the distribution of the value obtained w.r.t the generation at which the best value is found: 
- is there much variation?
- is there a trend (i.e. does increasing popsize result in a decrease at which the best solution is found?

In [ ]:
# code for calcualating and printing statistics from the dataframe df_Bests which  (from above) contains that best result per run per popsize
print(df_Bests.groupby('popsize').mean())

#  you can calculate other statistics such as median, std if you want
#print(df_Bests.groupby('popsize').median())
#print(df_Bests.groupby('popsize').std())

# plot the boxplot of best fitness per population size
boxplot = df_Bests.boxplot(column=['fitness'], by=['popsize'])

# plot genMaxFound per population size
boxplot = df_Bests.boxplot(column=['genMaxFound'], by=['popsize'])



##  Statistics

If we want to apply statistical tests, we need to extract the relevant data as lists - see the code below. Refer to the lecture for interpreting the statistical value. Based on the value - can we reject the null hpyothesis "**changing the population size from 50 to 150 has no effect on performance**"

In [ ]:
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind
from scipy.stats import shapiro

# select two subsets of fitness data, one where popsize = 50 and the other where popsize = 150
p50 = df_Bests.fitness[df_Bests.popsize == 50]
p150= df_Bests.genMaxFound[df_Bests.popsize == 150]

# we need to convert these subsets to lists to pass to the statistics functions
p50l= p50.tolist()
p150l = p150.tolist()

# check for normality
stat,p = shapiro(p50l)
print('p=50: normality p=%f' %(p))

if p<0.05:
    print("Reject null hypothesis (distribution is not normal)")
else:
    print("Cannot reject null hypothesis (assume distribution is normal)")
    
stat,p = shapiro(p150l)
print('p=150: normality p=%f' %(p))
if p<0.05:
    print("Reject null hypothesis (distribution is not normal)")
else:
    print("Cannot reject null hypothesis (assume distribution is normal)")


In [ ]:
#now apply the  appropriate statistical test depending on the result of the above
stat,p = ttest_ind(p50l,p150l)
#stat,p = mannwhitneyu(p50l,p150l)

print('p=%f' %(p))
if p<0.05:
    print("Reject null hypothesis - there is a significant difference in the experiments")
else:
    print("Cannot reject null hypothesis - no significant difference")

# Task 1: Exploring the effect of mutation rate

Fix all parameters except the mutation rate to values of your own choosing. Now run experiments over a range of mutation values from 0 to 1 in steps 0.2.  Create a boxplot to visualise the results and make sure you can explain what you observe.  

Select the mutation rate that appears to perform best, and the one that appears to perform worst. Conduct a signficance test to determine whether there is a statistically signifcant difference between the two mutation rates. Remember to first test for normality and then apply the appropriate test 

# Task 2: Exploring crossover types:

Now modify the code above to run the EA 10x using 1pt crossover, 2pt crossover and uniform crossover keeping all over parameters fixed.  (Set the crossover, run 10x and write the result to a dataframe that has a column noting the crossover type as well as one to record fitness. Now reset the crossover to the next one to be tested, repeat and add the results to the same dataframe).

Again plot the boxplot and then perform a statistical test between each pair of crossover operators to determine significance


# Task 3: Exploring effect of tournament size

Conduct and experiment to determine the influence of tournament size using a fixed population size of 100. Design the experiment, run it, plot the graphs and conduct a statistical test to determine whether it is possible to reject the null hypothesis "the tournament size has no effect on fitness when solving max-one"


# Task 4: Explore the effect of selection operator:

Design and conduct an experiment to evaluate the hypothesis "there is no difference between tournament selection size 2 and using roulette wheel (fitness proportionate selection).  You can test the latter using the inbuilt DEAP selection operator selRoulette()

# Finally
Please use any extra time to make sure you have completed the tutorials from previous weeks